# 📖 Notebook 1: A Stale Replica Returns Old Data

Replicas drift apart for many reasons: dropped writes, network blips, recovery from backup. If the coordinator naively returns the *first* response from any replica, clients can read **stale** values.


## 🛠️ Setup

```bash
cd 02-distributed-primitives/read-repair
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🟥 BAD: trust the first response

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, Tuple
import random

@dataclass
class Replica:
    name: str
    # value -> (value, timestamp). Higher timestamp wins.
    data: Dict[str, Tuple[str,int]] = field(default_factory=dict)
    def write(self, k, v, ts): self.data[k] = (v, ts)
    def read(self, k): return self.data.get(k)

r1, r2, r3 = Replica('r1'), Replica('r2'), Replica('r3')
# r1 and r2 have the latest write; r3 missed it.
r1.write('user:42', 'Alice v2', ts=200)
r2.write('user:42', 'Alice v2', ts=200)
r3.write('user:42', 'Alice v1', ts=100)

def naive_read(k):
    # Pick any replica; if it answers, we're done.
    chosen = random.choice([r1, r2, r3])
    return chosen.name, chosen.read(k)

random.seed(0)
for _ in range(6):
    print(naive_read('user:42'))


Whenever the coin lands on `r3`, the client gets stale `Alice v1`.

👉 Next: query *several* replicas, return the freshest, and **repair** the stale one in the background.